<a href="https://colab.research.google.com/github/MinsooKwak/RAG/blob/main/test/preprocessing/arvix_cat_extract_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Library**

In [1]:
!pip install -qU langchain-community arxiv pymupdf pdf2image pypdf PyPDF2

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.0/20.0 MB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 298.0/298.0 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.3/81.3 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 kB 3.0 MB/s eta 0:00:00


**카테고리별로 가져올 수 있는지 확인 (가능)**

- 카테고리 정보 : https://arxiv.org/category_taxonomy
- 검색어 search (web) : https://arxiv.org/search/

- 대분류 / 소분류별 가져오기 (따로 구성)
  - 우선 대분류 진행
  - 추후 고도화시 소분류 반영 (이는 어떤 카테고리에서 분석이 부족한지 반영할 수 있음)

In [ ]:
#major_categories = ["cs", "math", "physics", "eess", "econ", "q-bio", "q-fin", "stat", "astro-ph"] # 아래 수정

In [ ]:
import os
import pandas as pd
from langchain.document_loaders import ArxivLoader
from langchain.document_loaders import PyPDFLoader
from pdf2image import convert_from_path
from huggingface_hub import hf_hub_download
#import cv2
#from doclayout_yolo import YOLOv10
from PIL import Image

# ArxivLoader를 활용해 논문을 가져옵니다
loader = ArxivLoader(
    query= "cat : q-bio.*",               # 카테고리별 추출 가능 여부 Test  #f"cat:{category}"
    load_max_docs=50,               # 최대 문서 수
    load_all_available_meta=True    # 메타데이터 전체 로드 여부
)
docs = loader.load()

print(f"Loaded documents: {len(docs)}")


# 메타데이터 정보를 데이터프레임으로 저장
metadata_list = []
for doc in docs:
    metadata = doc.metadata
    metadata_list.append({
        'Title': metadata.get('Title', ''),
        'Summary': metadata.get('Summary', ''),
        'PDF Link': metadata.get('links', [])[-1] if metadata.get('links') else ''
    })

dataframe = pd.DataFrame(metadata_list)

# 데이터프레임 확인
print(dataframe)

Loaded documents: 3
                                   Title  \
0      Semistability and CAT(0) Geometry   
1                  Quantum Mona Lisa Cat   
2  On the Topological Complexity of Maps   

                                             Summary  \
0  We explain why semistability of a one-ended pr...   
1  Schr\"{o}dinger's Cat was proposed by Erwin Sc...   
2  We define and develop a homotopy invariant not...   

                            PDF Link  
0  http://arxiv.org/pdf/1703.07003v1  
1  http://arxiv.org/pdf/2001.10184v2  
2  http://arxiv.org/pdf/2011.10646v1  


In [ ]:
dataframe.head()

,Title,Summary,PDF Link
0,Semistability and CAT(0) Geometry,We explain why semistability of a one-ended pr...,http://arxiv.org/pdf/1703.07003v1
1,Quantum Mona Lisa Cat,"Schr\""{o}dinger's Cat was proposed by Erwin Sc...",http://arxiv.org/pdf/2001.10184v2
2,On the Topological Complexity of Maps,We define and develop a homotopy invariant not...,http://arxiv.org/pdf/2011.10646v1


In [ ]:
test_queries_lst = ["cat : cs.*", "cat : math.*", "cat : physics.*", "cat : eess.*", "cat : econ.*", "cat : q-bio.*", "cat : q-fin*", "cat : stat*", "cat : astro-ph.*"]
len(test_queries_lst)

9

In [ ]:
# 전체 카테고리(대분류) 추출 코드 (arxivloader편) - 미사용
import os
import pandas as pd
from langchain.document_loaders import ArxivLoader

# 쿼리 리스트 및 각 쿼리에 따른 최대 문서 수 설정
queries = ["cat : cs.*", "cat : math.*", "cat : physics.*", "cat : eess.*", "cat : econ.*", "cat : q-bio.*", "cat : q-fin*", "cat : stat*", "cat : astro-ph.*"]
load_max_docs_list = [5, 5, 5, 5, 5, 5, 5, 5, 5]  # 9개의 category이므로 우선 5개씩 저장해서 활용

# 결과를 저장할 리스트 초기화
metadata_list = []

# 각 쿼리에 대해 문서 로드 및 처리
for query, max_docs in zip(queries, load_max_docs_list):
    # ArxivLoader를 통해 문서 로드
    loader = ArxivLoader(
        query=query,                  # 현재 쿼리
        load_max_docs=max_docs,       # 최대 문서 수
        load_all_available_meta=True  # 메타데이터 전체 로드
    )
    docs = loader.load()

    # 각 문서에 대해 메타데이터 추출 및 저장
    for doc in docs:
        metadata = doc.metadata
        metadata_list.append({
            'Query': query,  # 쿼리 정보 추가
            'Title': metadata.get('Title', ''),
            'Summary': metadata.get('Summary', ''),
            'PDF Link': metadata.get('links', [])[-1] if metadata.get('links') else ''
        })

# 메타데이터 정보를 데이터프레임으로 변환
dataframe = pd.DataFrame(metadata_list)

# 데이터프레임 확인
print(dataframe)

# 데이터프레임 저장 (선택 사항)
output_path = "arxiv_metadata.csv"
dataframe.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f"Metadata saved to {output_path}")

               Query                                              Title  \
0         cat : cs.*  Barut-Girardello coherent states for sp(N,C) a...   
1         cat : cs.*  Demonstrating CAT: Synthesizing Data-Aware Con...   
2         cat : cs.*             Undirected Cat-and-Mouse is P-complete   
3       cat : math.*  Random groups have fixed points on CAT(0) cube...   
4       cat : math.*  Low-dimensional representations of matrix grou...   
5       cat : math.*                                       Hydra groups   
6    cat : physics.*  Is it possible to suspend the spread of an epi...   
7    cat : physics.*                         Free Will and Falling Cats   
8    cat : physics.*                    A Requiem For Schrödinger's Cat   
9       cat : eess.*  CAT: A CTC-CRF based ASR Toolkit Bridging the ...   
10      cat : eess.*                         CAT: CRF-based ASR Toolkit   
11      cat : eess.*  Assessing the Circadian Rhythm of Cats Living ...   
12      cat : econ.*  Seq

In [ ]:
# Langchain 기반 Loader 설정 확인
from langchain.document_loaders import ArxivLoader
#help(ArxivLoader)

#!pip show langchain

In [ ]:
# Test

import requests
import xml.etree.ElementTree as ET

# ArXiv API URL 구성
url = "http://export.arxiv.org/api/query"
params = {
    "search_query": "cat:q-bio.*",
    "start": 0,
    "max_results": 50
}
response = requests.get(url, params=params)

# API 응답 확인
print(response.status_code)  # 응답 코드 (200이어야 정상)
#print(response.text)         # 응답 내용 출력

# XML 응답 파싱
root = ET.fromstring(response.text)
entries = root.findall('{http://www.w3.org/2005/Atom}entry')

# 논문 메타데이터 추출
metadata_list = []
for entry in entries:
    title = entry.find('{http://www.w3.org/2005/Atom}title').text
    summary = entry.find('{http://www.w3.org/2005/Atom}summary').text
    pdf_link = None
    for link in entry.findall('{http://www.w3.org/2005/Atom}link'):
        if link.attrib.get('title') == 'pdf':
            pdf_link = link.attrib['href']
            break
    metadata_list.append({'Title': title, 'Summary': summary, 'PDF Link': pdf_link})

# 결과 출력
for meta in metadata_list:
    print(meta)

print(f'len : {len(metadata_list)}')

200
{'Title': 'Acto-myosin clusters as active units shaping living matter', 'Summary': '  Stress generation by the actin cytoskeleton shapes cells and tissues. Despite\nimpressive progress in live imaging and quantitative physical descriptions of\ncytoskeletal network dynamics, the connection between processes at molecular\nscales and cell-scale spatio-temporal patterns is still unclear. Here we review\nstudies reporting acto-myosin clusters of micrometer size and with lifetimes of\nseveral minutes in a large number of organisms ranging from fission yeast to\nhumans. Such structures have also been found in reconstituted systems in vitro\nand in theoretical analysis of cytoskeletal dynamics. We propose that tracking\nthese clusters can serve as a simple readout for characterising living matter.\nSpatio-temporal patterns of clusters could serve as determinants of\nmorphogenetic processes that play similar roles in diverse organisms.\n', 'PDF Link': 'http://arxiv.org/pdf/2408.05119v1'}
{'

50

In [ ]:
df_bio_metadata_list = pd.DataFrame(metadata_list)
print(len(df_bio_metadata_list))
df_bio_metadata_list.head()

50


,Title,Summary,PDF Link
0,Acto-myosin clusters as active units shaping l...,Stress generation by the actin cytoskeleton ...,http://arxiv.org/pdf/2408.05119v1
1,Genome-scale reconstruction of the metabolic n...,"The gram-negative bacterium Yersinia pestis,...",http://arxiv.org/pdf/0903.4219v1
2,Infer metabolic velocities from moment differe...,Metabolic pathways are fundamental maps in b...,http://arxiv.org/pdf/2402.14887v4
3,Spatial cancer systems biology resolves hetero...,Spatially annotated single-cell datasets pro...,http://arxiv.org/pdf/2303.00933v1
4,Prediction of cellular burden with host-circui...,Heterologous gene expression draws resources...,http://arxiv.org/pdf/2004.00995v2


In [2]:
#!pip install -qU langchain-community arxiv pymupdf pdf2image pypdf PyPDF2

In [56]:
# 전체 카테고리 (API 활용 방식)
import os
import pandas as pd
from langchain.document_loaders import ArxivLoader
from langchain.document_loaders import PyPDFLoader
from pdf2image import convert_from_path
from huggingface_hub import hf_hub_download
#import cv2
#from doclayout_yolo import YOLOv10
from PIL import Image
import time
import requests
import xml.etree.ElementTree as ET
from PyPDF2 import PdfReader
import io

# ArXiv API URL 구성
url = "http://export.arxiv.org/api/query"

# Query 리스트와 최대 문서 수 정의
# 쿼리 리스트 및 각 쿼리에 따른 최대 문서 수 설정
queries = ["cat : cs.*", "cat : math.*", "cat : physics.*", "cat : eess.*", "cat : econ.*", "cat : q-bio.*", "cat : q-fin.*", "cat : stat.*", "cat : astro-ph.*"]
load_max_docs_list = [5, 5, 5, 5, 5, 5, 5, 5, 5]  # 9개의 category이므로 우선 5개씩 저장해서 활용

#### 본문 저장하기 위한 함수 (PdfReader)
"""
def extract_text_from_pdf(pdf_url):
    try:
        response = requests.get(pdf_url)
        response.raise_for_status()  # 요청 성공 여부 확인

        with io.BytesIO(response.content) as pdf_file:
            reader = PdfReader(pdf_file)
            text = ""
            for page in reader.pages:
                text += page.extract_text()
        return text
    except Exception as e:
        print(f"Error extracting text from {pdf_url}: {e}")
        return None
"""
#### 본문 반영 (arxivloader 반영)
def apply_arxiv_content(row):
    '''
    아래 metadata에서 본문 반영(arxivloader 사용)
    '''
    try:
        loader = ArxivLoader(query=row, load_max_docs=1, load_all_available_meta=True)
        docs = loader.load()
        time.sleep(1)
        if docs:
            return docs[0].metadata  # 메타데이터만 반환
    except Exception as e:
        print(f"Error loading metadata for '{title}': {e}")
        return None

# 메타데이터 저장
metadata_list = []

for query, max_docs in zip(queries, load_max_docs_list):
    params = {
        "search_query": query,
        "start": 0,
        "max_results": max_docs
    }

    response = requests.get(url, params=params)

    # API 응답 확인
    if response.status_code != 200:
        print(f"Error: Failed to fetch data for query '{query}' with status code {response.status_code}")
        continue

    # XML 응답 파싱
    root = ET.fromstring(response.text)
    entries = root.findall('{http://www.w3.org/2005/Atom}entry')

    # 논문 메타데이터 추출
    category = query.split(':')[1].split('.')[0]  # 'cat:' 이후, '.' 이전 부분 추출

    # 논문 메타데이터 추출
    for entry in entries:
        title = entry.find('{http://www.w3.org/2005/Atom}title').text
        summary = entry.find('{http://www.w3.org/2005/Atom}summary').text
        pdf_link = None
        for link in entry.findall('{http://www.w3.org/2005/Atom}link'):
            if link.attrib.get('title') == 'pdf':
                pdf_link = link.attrib['href']
                break

        metadata_list.append({
            'Category': category,
            'Query': query,
            'Title': title,
            'Summary': summary,
            'PDF Link': pdf_link,
        })

        # PDF 본문 추출 (PdfReader 일 때)
        #pdf_text = extract_text_from_pdf(pdf_link) if pdf_link else None

# 결과 출력
for meta in metadata_list:
    print(meta)

print(f'Total number of papers fetched: {len(metadata_list)}')

{'Category': ' cs', 'Query': 'cat : cs.*', 'Title': 'Brittle System Analysis', 'Summary': '  The goal of this paper is to define and analyze systems which exhibit brittle\nbehavior. This behavior is characterized by a sudden and steep decline in\nperformance as the system approaches the limits of tolerance. This can be due\nto input parameters which exceed a specified input, or environmental conditions\nwhich exceed specified operating boundaries. An analogy is made between brittle\ncommmunication systems in particular and materials science.\n', 'PDF Link': 'http://arxiv.org/pdf/cs/9904016v1'}
{'Category': ' cs', 'Query': 'cat : cs.*', 'Title': 'The Unix KISS: A Case Study', 'Summary': '  In this paper we show that the initial philosophy used in designing and\ndeveloping UNIX in early times has been forgotten due to "fast practices". We\nquestion the leitmotif that microkernels, though being by design adherent to\nthe KISS principle, have a number of context switches higher than their\

In [57]:
metadata_final = pd.DataFrame(metadata_list)
metadata_final['Title'] = metadata_final['Title'].astype(str)
metadata_final

,Category,Query,Title,Summary,PDF Link
0,cs,cat : cs.*,Brittle System Analysis,The goal of this paper is to define and anal...,http://arxiv.org/pdf/cs/9904016v1
1,cs,cat : cs.*,The Unix KISS: A Case Study,In this paper we show that the initial philo...,http://arxiv.org/pdf/cs/0701021v2
2,cs,cat : cs.*,ASC-Hook: fast and transparent system call hoo...,Intercepting system calls is crucial for too...,http://arxiv.org/pdf/2412.05784v3
3,cs,cat : cs.*,Learnings from an Under the Hood Analysis of a...,Conventional object-stores are built on top ...,http://arxiv.org/pdf/2207.01849v1
4,cs,cat : cs.*,An Example of Clifford Algebras Calculations w...,This example of Clifford algebras calculatio...,http://arxiv.org/pdf/cs/0410044v5
5,math,cat : math.*,Generalized spaces for constructive algebra,The purpose of this contribution is to give ...,http://arxiv.org/pdf/2012.13850v1
6,math,cat : math.*,Constructing projective modules,We discuss elements of a social history of t...,http://arxiv.org/pdf/2412.05250v1
7,math,cat : math.*,Quantalic spectra of semirings,Spectrum constructions appear throughout mat...,http://arxiv.org/pdf/2201.06408v1
8,math,cat : math.*,Von Neumann coordinatization is not first-order,"A lattice L is coordinatizable, if it is iso...",http://arxiv.org/pdf/math/0409250v3
9,math,cat : math.*,Analysis in J_2,This is an expository paper in which I expla...,http://arxiv.org/pdf/math/0509245v2


In [58]:
metadata_final['Content'] = metadata_final['Title'].apply(apply_arxiv_content)
metadata_final

MuPDF error: syntax error: unknown keyword: 'ca'

MuPDF error: syntax error: unknown keyword: 'ca'

MuPDF error: syntax error: unknown keyword: 'ca'

MuPDF error: syntax error: unknown keyword: 'ca'

MuPDF error: syntax error: unknown keyword: 'ca'

MuPDF error: syntax error: unknown keyword: 'ca'

MuPDF error: syntax error: unknown keyword: 'ca'

MuPDF error: syntax error: unknown keyword: 'ca'

MuPDF error: syntax error: unknown keyword: 'ca'

MuPDF error: syntax error: unknown keyword: 'ca'

MuPDF error: syntax error: unknown keyword: 'ca'

MuPDF error: syntax error: unknown keyword: 'ca'

MuPDF error: syntax error: unknown keyword: 'ca'

MuPDF error: syntax error: unknown keyword: 'ca'

MuPDF error: syntax error: unknown keyword: 'ca'

MuPDF error: syntax error: unknown keyword: 'ca'

MuPDF error: syntax error: unknown keyword: 'ca'

MuPDF error: syntax error: unknown keyword: 'ca'

MuPDF error: syntax error: unknown keyword: 'ca'

MuPDF error: syntax error: unknown keyword: 'ca'



,Category,Query,Title,Summary,PDF Link,Content
0,cs,cat : cs.*,Brittle System Analysis,The goal of this paper is to define and anal...,http://arxiv.org/pdf/cs/9904016v1,arXiv:cs/9904016v1 [cs.NI] 22 Apr 1999\nBUSH...
1,cs,cat : cs.*,The Unix KISS: A Case Study,In this paper we show that the initial philo...,http://arxiv.org/pdf/cs/0701021v2,None
2,cs,cat : cs.*,ASC-Hook: fast and transparent system call hoo...,Intercepting system calls is crucial for too...,http://arxiv.org/pdf/2412.05784v3,ASC-Hook: fast and transparent system call hoo...
3,cs,cat : cs.*,Learnings from an Under the Hood Analysis of a...,Conventional object-stores are built on top ...,http://arxiv.org/pdf/2207.01849v1,None
4,cs,cat : cs.*,An Example of Clifford Algebras Calculations w...,This example of Clifford algebras calculatio...,http://arxiv.org/pdf/cs/0410044v5,arXiv:cs/0410044v5 [cs.MS] 9 Dec 2005\narXiv...
5,math,cat : math.*,Generalized spaces for constructive algebra,The purpose of this contribution is to give ...,http://arxiv.org/pdf/2012.13850v1,arXiv:alg-geom/9708025v1 29 Aug 1997\nDuality...
6,math,cat : math.*,Constructing projective modules,We discuss elements of a social history of t...,http://arxiv.org/pdf/2412.05250v1,arXiv:1601.02737v1 [math.RT] 12 Jan 2016\nTH...
7,math,cat : math.*,Quantalic spectra of semirings,Spectrum constructions appear throughout mat...,http://arxiv.org/pdf/2201.06408v1,Q U A N TA L E S A N D H Y P E R S T R U C T U...
8,math,cat : math.*,Von Neumann coordinatization is not first-order,"A lattice L is coordinatizable, if it is iso...",http://arxiv.org/pdf/math/0409250v3,arXiv:1312.1663v1 [math.RA] 5 Dec 2013\nFrom...
9,math,cat : math.*,Analysis in J_2,This is an expository paper in which I expla...,http://arxiv.org/pdf/math/0509245v2,arXiv:cond-mat/9706078v1 [cond-mat.stat-mech]...


In [52]:
metadata_final.loc[32]['Title']

'Pricing Financial Derivatives Subject to Counterparty Risk and Credit\n  Value Adjustment'

In [51]:
#### 오류 확인
apply_arxiv_content(metadata_final.loc[32])

Error loading metadata for 'Roles and Needs of Laboratory Astrophysics in NASA's Space and Earth
  Science Mission': 'Series' object has no attribute 'split'


- 실험 결과 case
  - None 발생되는 케이스
  - 결과값 논문 제목 들어있는 케이스
  - 논문 pdf 링크 들어있는 케이스 등 있음
  
- 결론
  - pdf 유형이 다를 경우의 가능성 있음.  
  - 대안 시도 가능 방안 : 그냥 pypdf 갖고 와서 텍스트 불러오고 전처리, abstract, 본론, 결론 등 나눠서 pdf 저장하는 방안.
  - 요약하고 결합하는 방식을 활용해야 하지 않을까 (요약시)
  - 요약 아닌 Q&A 경우 사용자 쿼리 따라 질의 유형 구분하고 적합한 위치 선정해 찾는 방안을 택해야 하지 않을까

### 디렉토리 생성해 데이터 추출

- 데이터 확보건

In [ ]:
#### 아직 미진행....
#### 여기서 텍스트, 이미지, figure 추출하고 저장
#### 아래는 프롬프트로 LLM 통해 텍스트화 (프롬프트 고도화도 포함)
#### 벡터 DB 저장
#### 벡터 DB 저장 내용 가져와 사용 방안
#### 각 유형 따라 결과 도출

- Test 위한 데이터 확보건

In [ ]:
#### 위 df 확보시 Q&A, GT 생성
#### Test (RAGAS / LLM as a Judge / langchain evaluate)

- 추가 실험 건

In [ ]:
#### LLM >>> sLM
#### 오픈소스, 비용 낮춰도 가능하게
#### 프롬프틑 고도화

### Test 전 Demo

**기존 쿼리 기반 추출 코드(Basic) - Metadata**

In [ ]:
"""
import os
import pandas as pd
from langchain.document_loaders import ArxivLoader
from pdf2image import convert_from_path
from huggingface_hub import hf_hub_download
import cv2
from doclayout_yolo import YOLOv10
from PIL import Image

# ArxivLoader를 활용해 논문을 가져옵니다
loader = ArxivLoader(
    query="Agent",
    load_max_docs=20,              # 최대 문서 수
    load_all_available_meta=True   # 메타데이터 전체 로드 여부
)
docs = loader.load()

# 메타데이터 정보를 데이터프레임으로 저장
metadata_list = []
for doc in docs:
    metadata = doc.metadata
    metadata_list.append({
        'Title': metadata.get('Title', ''),
        'Summary': metadata.get('Summary', ''),
        'PDF Link': metadata.get('links', [])[-1] if metadata.get('links') else ''
    })

dataframe = pd.DataFrame(metadata_list)

# 데이터프레임 확인
print(dataframe)

# DocLayout-YOLO 및 필수 패키지 설치
os.system("pip install git+https://github.com/opendatalab/DocLayout-YOLO.git")
os.system("pip install pdf2image huggingface_hub")
os.system("sudo apt-get install poppler-utils")

# PDF 다운로드 함수
def download_pdf(pdf_url, filename):
    import requests
    response = requests.get(pdf_url)
    with open(filename, 'wb') as f:
        f.write(response.content)
    print(f"Downloaded: {filename}")

# PDF를 이미지로 변환
def convert_pdf_to_images(pdf_path, output_dir):
    try:
        os.makedirs(output_dir, exist_ok=True)
        images = convert_from_path(pdf_path)
        image_paths = []
        for i, image in enumerate(images):
            image_path = os.path.join(output_dir, f"page_{i+1}.jpg")
            image.save(image_path, "JPEG")
            image_paths.append(image_path)
        print(f"Converted {pdf_path} to images in {output_dir}")
        return image_paths
    except Exception as e:
        print(f"Error converting {pdf_path} to images: {e}")
        return []

# DocLayout-YOLO를 활용한 디텍션 및 저장

def extract_content_with_doclayout_yolo(image_paths, output_dir, pdf_index):
    os.makedirs(output_dir, exist_ok=True)

    # Pre-trained 모델 다운로드 및 로드
    model_path = hf_hub_download(repo_id="juliozhao/DocLayout-YOLO-DocStructBench", filename="doclayout_yolo_docstructbench_imgsz1024.pt")
    model = YOLOv10(model_path)

    metadata = []

    for page_number, image_path in enumerate(image_paths, start=1):
        # 모델 예측
        det_res = model.predict(
            source=image_path,   # 이미지 경로
            imgsz=1024,          # 예측 이미지 크기
            conf=0.2,            # 신뢰도 임계값
            device="cpu"         # CPU에서 실행
        )

        # 결과 필터링 및 저장
        for det in det_res[0].boxes:
            bbox = det.xyxy.numpy().flatten()  # 바운딩 박스 좌표 추출
            cls = int(det.cls.cpu().numpy())  # 클래스 인덱스 추출
            label = model.names[cls]

            if label in ["table","table_caption","table_footnote", "figure", "figure_caption","isolate_formula","formula_caption"]:
                image = Image.open(image_path)
                cropped = image.crop((bbox[0], bbox[1], bbox[2], bbox[3]))

                output_label_dir = os.path.join(output_dir, label + "s")
                os.makedirs(output_label_dir, exist_ok=True)
                output_image_path = os.path.join(output_label_dir, os.path.basename(image_path).replace(".jpg", f"_{label}.jpg"))
                cropped.save(output_image_path)
                print(f"Saved {label} to {output_image_path}")

                metadata.append({
                    "pdf_index": pdf_index,
                    "page_number": page_number,
                    "label": label,
                    "bbox": bbox.tolist(),
                    "output_path": output_image_path
                })

    return metadata

# PDF 다운로드 및 콘텐츠 추출 함수
def save_pdfs_and_extract_content(dataframe):
    extracted_data_dir = "extracted_data"
    os.makedirs(extracted_data_dir, exist_ok=True)

    all_metadata = []

    for index, row in dataframe.iterrows():
        pdf_link = row['PDF Link']
        title = row['Title']
        if pdf_link:
            pdf_filename = os.path.join(extracted_data_dir, f"paper_{index+1}.pdf")
            download_pdf(pdf_link, pdf_filename)

            # PDF를 이미지로 변환
            image_output_dir = os.path.join(extracted_data_dir, f"paper_{index+1}_images")
            image_paths = convert_pdf_to_images(pdf_filename, image_output_dir)

            if image_paths:
                # DocLayout-YOLO 디텍션 및 결과 저장
                content_output_dir = os.path.join(extracted_data_dir, f"paper_{index+1}_content")
                metadata = extract_content_with_doclayout_yolo(image_paths, content_output_dir, pdf_index=index+1)
                all_metadata.extend(metadata)
                print(f"Extracted content from {pdf_filename} saved to {content_output_dir}")

    # 메타데이터를 CSV로 저장
    metadata_df = pd.DataFrame(all_metadata)
    metadata_df.to_csv(os.path.join(extracted_data_dir, "detection_metadata.csv"), index=False)
    print(f"Detection metadata saved to {os.path.join(extracted_data_dir, 'detection_metadata.csv')}.")

# 함수 실행
save_pdfs_and_extract_content(dataframe)
"""